[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VaishnaviJagtap18/42-days-aiml-challenge/blob/main/week4_deep_learning/day29_rnns_lstms/day29_notebook.ipynb)

# Day 29 / 42: RNNs and LSTMs
### #42DaysOfML | Week 4: Deep Learning

---

## What You'll Learn
- How RNNs process sequences (and why the "loop" matters)
- The vanishing gradient problem — proven with actual gradients
- How LSTM gates (forget, input, output) fix this
- Build an LSTM for next-word prediction on a text corpus
- Build a sentiment classifier with LSTM
- When to use LSTMs vs Transformers in 2024
- Production problem: sequence length at inference time

---

In [ ]:
!pip install torch matplotlib numpy --quiet

## The Concept

A feedforward neural network processes one input and produces one output. It has no memory of previous inputs. This works for classifying a single image. It doesn't work for predicting the next word in a sentence, because the meaning of a word depends on what came before it.

**RNNs (Recurrent Neural Networks)** add a hidden state that gets passed from one time step to the next. At each step, the network sees the current input AND the previous hidden state. The hidden state is the network's "memory" of what it has seen so far.

```
h_t = tanh(W_h * h_{t-1} + W_x * x_t + b)
```

**The vanishing gradient problem:**
During backpropagation through time (BPTT), gradients must flow back through every time step. Each step multiplies the gradient by the weight matrix's values. If these values are <1 (which they usually are after tanh), the gradient shrinks exponentially. After 20-30 steps, the gradient reaching the early time steps is essentially zero. The network cannot learn long-range dependencies.

**LSTMs (Long Short-Term Memory)** fix this by adding a **cell state**: a separate memory channel that runs through the entire sequence with only additive updates (no multiplicative shrinking). Three gates control what gets into and out of the cell state:

- **Forget gate** f_t: Which information from the previous cell state to erase. Output ∈ [0,1]. 0 = forget completely, 1 = keep completely.
- **Input gate** i_t: Which new information to write into the cell state.
- **Output gate** o_t: What to read from the cell state as the hidden state for this time step.

The cell state update is additive (not multiplicative), which allows gradients to flow back thousands of steps without vanishing. This is why LSTMs can learn that the subject of a sentence (word 3) is the antecedent of a pronoun (word 47).

**Where LSTMs are still used in production (2024):**
- Time series forecasting (stock prices, energy demand, sensor data)
- Audio processing (speech recognition preprocessing)
- Anomaly detection on sequential logs
- On-device NLP where Transformers are too heavy

In [ ]:
# ============================================================
# SECTION 1: Prove the Vanishing Gradient Problem
# Show gradient magnitude shrinking across time steps
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

def tanh(x):
    return np.tanh(x)

def tanh_derivative(x):
    return 1 - np.tanh(x)**2

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# Simulate gradient flowing backward through T time steps in a vanilla RNN
# At each step, gradient is multiplied by: W_h * tanh'(h_t)
# tanh'(x) is at most 1.0 (at x=0), typically 0.2-0.5 in practice

T = 50  # sequence length
np.random.seed(42)

# Scenario 1: W_h = 0.9 (slightly below 1) — common initialisation
# Scenario 2: W_h = 1.0 — exploding gradient risk
# Scenario 3: W_h = 0.5 — very common case

def compute_gradient_flow(W_h, T, tanh_deriv_value=0.5):
    """Simulate gradient magnitude at each time step during BPTT."""
    grad = 1.0  # start with gradient = 1 at final step
    gradients = [grad]
    for t in range(T - 1):
        # Each step: grad = grad * W_h * tanh'(h_t)
        # tanh' is bounded by 1, but in practice ~0.5 for typical activations
        grad = grad * abs(W_h) * tanh_deriv_value
        gradients.append(grad)
    return gradients[::-1]  # reverse: step 0 first

configs = [
    (0.9, '#F44336', 'W=0.9, tanh\'=0.5 (typical case)'),
    (1.0, '#FF9800', 'W=1.0, tanh\'=0.5 (borderline)'),
    (0.5, '#9C27B0', 'W=0.5, tanh\'=0.5 (fast vanishing)'),
    (1.2, '#4CAF50', 'W=1.2, tanh\'=0.5 (exploding!)'),
]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for W, color, label in configs:
    grads = compute_gradient_flow(W, T)
    axes[0].plot(range(T), grads, label=label, color=color, linewidth=2)
    axes[1].semilogy(range(T), [max(g, 1e-50) for g in grads], label=label, color=color, linewidth=2)

for ax in axes:
    ax.set_xlabel('Time Step (0 = earliest, 49 = latest)', fontsize=11)
    ax.set_ylabel('Gradient Magnitude', fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='black', linewidth=0.5)

axes[0].set_title('Gradient Flow in Vanilla RNN\n(Linear scale)', fontsize=12, fontweight='bold')
axes[1].set_title('Gradient Flow in Vanilla RNN\n(Log scale — see how fast it vanishes)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

grads_09 = compute_gradient_flow(0.9, T)
print(f"Gradient at step 0 (50 steps back):  {grads_09[0]:.2e}")
print(f"Gradient at step 10 (40 steps back): {grads_09[10]:.2e}")
print(f"Gradient at step 40 (10 steps back): {grads_09[40]:.4f}")
print(f"Gradient at step 49 (final step):    {grads_09[49]:.4f}")
print(f"\nWith W=0.9: gradient shrinks {grads_09[49]/max(grads_09[0], 1e-50):.0e}x from step 49 to step 0.")
print("The network literally cannot learn anything that happened more than ~20 steps ago.")

In [ ]:
# ============================================================
# SECTION 2: LSTM Gates — Manual Implementation
# Implement one LSTM step from scratch to understand the math
# ============================================================
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

class ManualLSTMCell:
    """One step of an LSTM cell, implemented from the equations."""
    
    def __init__(self, input_size, hidden_size):
        self.hidden_size = hidden_size
        np.random.seed(42)
        scale = 0.1
        
        # LSTM has 4 weight matrices: for forget, input, cell, output gates
        # Each takes [x_t, h_{t-1}] as input (concatenated)
        self.Wf = np.random.randn(hidden_size, input_size + hidden_size) * scale
        self.Wi = np.random.randn(hidden_size, input_size + hidden_size) * scale
        self.Wc = np.random.randn(hidden_size, input_size + hidden_size) * scale
        self.Wo = np.random.randn(hidden_size, input_size + hidden_size) * scale
        
        self.bf = np.ones(hidden_size)    # forget gate bias initialised to 1 (remember by default)
        self.bi = np.zeros(hidden_size)
        self.bc = np.zeros(hidden_size)
        self.bo = np.zeros(hidden_size)
    
    def step(self, x_t, h_prev, c_prev):
        """
        One LSTM forward step.
        
        x_t:    input at current time step (input_size,)
        h_prev: previous hidden state (hidden_size,)
        c_prev: previous cell state (hidden_size,)
        
        Returns: (h_t, c_t, gate_values)
        """
        # Concatenate input and previous hidden state
        combined = np.concatenate([x_t, h_prev])
        
        # Forget gate: what to erase from cell state
        f_t = sigmoid(self.Wf @ combined + self.bf)
        
        # Input gate: how much of the new candidate to write
        i_t = sigmoid(self.Wi @ combined + self.bi)
        
        # Candidate cell state: what to potentially write
        c_hat = np.tanh(self.Wc @ combined + self.bc)
        
        # Output gate: what to read from cell state as hidden state
        o_t = sigmoid(self.Wo @ combined + self.bo)
        
        # New cell state: forget some old info, add some new info (ADDITIVE!)
        c_t = f_t * c_prev + i_t * c_hat
        
        # New hidden state
        h_t = o_t * np.tanh(c_t)
        
        return h_t, c_t, {'forget': f_t, 'input': i_t, 'candidate': c_hat, 'output': o_t}


# Run a sequence through the LSTM cell and track gate activations
cell = ManualLSTMCell(input_size=4, hidden_size=8)

h = np.zeros(8)
c = np.zeros(8)

# Simulate a sequence of 10 time steps
T_demo = 10
forget_history = []
input_history = []
output_history = []

for t in range(T_demo):
    x = np.random.randn(4)  # random input at each step
    h, c, gates = cell.step(x, h, c)
    forget_history.append(gates['forget'].mean())
    input_history.append(gates['input'].mean())
    output_history.append(gates['output'].mean())

fig, ax = plt.subplots(figsize=(12, 5))
x_steps = range(1, T_demo + 1)
ax.plot(x_steps, forget_history, 'o-', color='#F44336', linewidth=2, markersize=8, label='Forget Gate (avg)')
ax.plot(x_steps, input_history, 's-', color='#2196F3', linewidth=2, markersize=8, label='Input Gate (avg)')
ax.plot(x_steps, output_history, '^-', color='#4CAF50', linewidth=2, markersize=8, label='Output Gate (avg)')
ax.axhline(y=0.5, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='0.5 threshold')
ax.set_xlabel('Time Step', fontsize=12)
ax.set_ylabel('Average Gate Activation (0=closed, 1=open)', fontsize=11)
ax.set_title('LSTM Gate Activations Across a Sequence\n(trained gates would show more structured patterns)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, 1.1)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Gate values are all in [0,1] (sigmoid output).")
print("Forget gate = 1.0: keep 100% of previous cell state")
print("Forget gate = 0.0: erase 100% of previous cell state")
print("Input gate = 0.0: don't write anything new to cell state")
print("\nThe cell state update c_t = f_t * c_prev + i_t * c_hat is ADDITIVE.")
print("This is what prevents vanishing gradients — no repeated multiplication.")

In [ ]:
# ============================================================
# SECTION 3: LSTM for Next-Word Prediction with PyTorch
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Small text corpus — enough to demonstrate the concept
corpus = """
machine learning is a subset of artificial intelligence that enables systems to learn from data
deep learning uses neural networks with many layers to learn representations from data
convolutional neural networks are used for image classification and object detection
recurrent neural networks process sequential data like text and time series
long short term memory networks solve the vanishing gradient problem in recurrent networks
transformers use self attention mechanisms and have replaced recurrent networks for many tasks
natural language processing enables machines to understand and generate human language
transfer learning allows models to apply knowledge from one domain to another domain
gradient descent optimizes neural network weights by following the negative gradient of the loss
overfitting occurs when a model performs well on training data but poorly on new data
regularization techniques like dropout and weight decay help prevent overfitting in neural networks
batch normalization stabilizes training by normalizing layer inputs across a mini batch
the attention mechanism allows models to focus on relevant parts of the input sequence
reinforcement learning trains agents to make decisions by rewarding correct actions
data augmentation creates new training examples by applying transformations to existing data
model evaluation requires separate training validation and test sets to avoid data leakage
feature engineering transforms raw data into representations that improve model performance
cross validation provides more reliable performance estimates by using multiple train test splits
hyperparameter tuning finds the best model configuration using grid search or random search
ensemble methods combine multiple models to produce better predictions than any single model
""".lower().split()

# Build vocabulary
word_counts = Counter(corpus)
vocab = ['<PAD>', '<UNK>'] + [w for w, c in word_counts.most_common()]
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
VOCAB_SIZE = len(vocab)

print(f"Corpus size:   {len(corpus)} tokens")
print(f"Vocabulary:    {VOCAB_SIZE} unique words")
print(f"Top 10 words:  {[w for w, _ in word_counts.most_common(10)]}")

# Create sequences: input = n words, target = next word
SEQ_LEN = 6

def text_to_tensor(words, word2idx):
    return [word2idx.get(w, word2idx['<UNK>']) for w in words]

indexed = text_to_tensor(corpus, word2idx)

sequences, targets = [], []
for i in range(len(indexed) - SEQ_LEN):
    sequences.append(indexed[i:i+SEQ_LEN])
    targets.append(indexed[i+SEQ_LEN])

X = torch.tensor(sequences, dtype=torch.long)
y = torch.tensor(targets, dtype=torch.long)

dataset = torch.utils.data.TensorDataset(X, y)
train_size = int(0.8 * len(dataset))
train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, len(dataset)-train_size])

trainloader = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True)
valloader = torch.utils.data.DataLoader(val_ds, batch_size=32)

print(f"\nSequences created: {len(sequences)}")
print(f"Training pairs:    {len(train_ds)}")
print(f"Validation pairs:  {len(val_ds)}")
print(f"\nExample: '{' '.join(corpus[:SEQ_LEN])}' -> '{corpus[SEQ_LEN]}'")

In [ ]:
# ============================================================
# Define and Train the LSTM Language Model
# ============================================================

class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout=0.3):
        super().__init__()
        
        # Embedding: convert token indices to dense vectors
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        # LSTM: num_layers stacked LSTM layers
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True  # input shape: (batch, seq_len, features)
        )
        
        # Output layer: predict probability over vocabulary
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x):
        # x: (batch, seq_len)
        embedded = self.dropout(self.embedding(x))  # (batch, seq_len, embed_dim)
        lstm_out, (h_n, c_n) = self.lstm(embedded)  # lstm_out: (batch, seq_len, hidden_dim)
        last_hidden = lstm_out[:, -1, :]             # take last time step (batch, hidden_dim)
        out = self.fc(self.dropout(last_hidden))      # (batch, vocab_size)
        return out


model = LSTMLanguageModel(
    vocab_size=VOCAB_SIZE,
    embed_dim=64,
    hidden_dim=128,
    num_layers=2,
    dropout=0.3
).to(device)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Architecture:")
print(model)

optimizer = optim.Adam(model.parameters(), lr=0.002)
criterion = nn.CrossEntropyLoss()
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

EPOCHS = 50
train_losses, val_losses = [], []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for x_batch, y_batch in trainloader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        output = model(x_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # gradient clipping
        optimizer.step()
        total_loss += loss.item()
    
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for x_batch, y_batch in valloader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            val_loss += criterion(model(x_batch), y_batch).item()
    
    train_losses.append(total_loss / len(trainloader))
    val_losses.append(val_loss / max(len(valloader), 1))
    scheduler.step()
    
    if (epoch + 1) % 10 == 0:
        perplexity = np.exp(val_losses[-1])
        print(f"Epoch {epoch+1:3d}/{EPOCHS}: Train Loss {train_losses[-1]:.3f} | "
              f"Val Loss {val_losses[-1]:.3f} | Perplexity {perplexity:.1f}")

print("\nPerplexity = exp(cross-entropy loss). Lower = better.")
print("Random guessing perplexity = vocab_size = ", VOCAB_SIZE)

In [ ]:
# ============================================================
# Generate Text with the Trained LSTM
# ============================================================

def generate_text(model, seed_words, num_words=10, temperature=1.0):
    """
    Generate text by predicting one word at a time.
    Temperature controls randomness: lower = more predictable, higher = more creative.
    """
    model.eval()
    words = seed_words[:]
    
    with torch.no_grad():
        for _ in range(num_words):
            # Use last SEQ_LEN words as context
            context = words[-SEQ_LEN:]
            # Pad if shorter than SEQ_LEN
            if len(context) < SEQ_LEN:
                context = ['<PAD>'] * (SEQ_LEN - len(context)) + context
            
            x = torch.tensor([[word2idx.get(w, word2idx['<UNK>']) for w in context]],
                              dtype=torch.long).to(device)
            
            logits = model(x)[0]  # (vocab_size,)
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            
            # Sample from probability distribution
            next_idx = torch.multinomial(probs, 1).item()
            words.append(idx2word[next_idx])
    
    return ' '.join(words)


print("TEXT GENERATION WITH TRAINED LSTM")
print("=" * 60)

seed_phrases = [
    ['machine', 'learning', 'is', 'a', 'subset', 'of'],
    ['recurrent', 'neural', 'networks', 'process', 'sequential', 'data'],
    ['gradient', 'descent', 'optimizes', 'neural', 'network', 'weights'],
]

for temp in [0.5, 1.0, 1.5]:
    print(f"\nTemperature = {temp} ({'more predictable' if temp < 1 else 'more random' if temp > 1 else 'balanced'}):")
    for seed in seed_phrases:
        generated = generate_text(model, seed[:], num_words=8, temperature=temp)
        print(f"  Seed: '{' '.join(seed)}'")
        print(f"  Generated: '{generated}'")
        print()

# Plot training curves
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_losses, label='Train Loss', color='#2196F3', linewidth=2)
ax.plot(val_losses, label='Val Loss', color='#F44336', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('LSTM Language Model Training Curve', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SECTION 4: LSTM for Sentiment Classification
# A more practical production task than language modelling
# ============================================================

# Minimal sentiment dataset — enough to show the pipeline
# In production you'd use SST-2, IMDB (25k reviews), or your own data
sentiment_data = [
    ("this model performs exceptionally well on all test cases", 1),
    ("the training loss decreases steadily with good convergence", 1),
    ("outstanding accuracy achieved after hyperparameter tuning", 1),
    ("the model generalizes well to unseen data across domains", 1),
    ("excellent performance on both precision and recall metrics", 1),
    ("great improvement in results after adding regularization techniques", 1),
    ("the neural network learned meaningful feature representations", 1),
    ("model achieves state of the art results on benchmark datasets", 1),
    ("the training pipeline is efficient and well documented", 1),
    ("validation accuracy improved significantly after data augmentation", 1),
    ("this model fails to converge and shows random predictions", 0),
    ("terrible performance with high loss after many training epochs", 0),
    ("the model overfits severely and cannot generalize to new data", 0),
    ("poor results on test set despite good training performance", 0),
    ("accuracy dropped significantly after deployment in production", 0),
    ("the model produces wrong predictions on most test samples", 0),
    ("training diverged and loss became not a number early on", 0),
    ("completely useless model that predicts the same class always", 0),
    ("very slow convergence and poor final accuracy on validation", 0),
    ("catastrophic forgetting occurred after transfer learning attempt", 0),
]

# Build vocabulary from sentiment data
all_words = []
for text, _ in sentiment_data:
    all_words.extend(text.lower().split())

sent_vocab = ['<PAD>', '<UNK>'] + list(set(all_words))
sent_word2idx = {w: i for i, w in enumerate(sent_vocab)}
SENT_VOCAB_SIZE = len(sent_vocab)

MAX_LEN = 12

def encode_sentence(text, word2idx, max_len):
    tokens = text.lower().split()[:max_len]
    encoded = [word2idx.get(w, word2idx['<UNK>']) for w in tokens]
    padded = encoded + [0] * (max_len - len(encoded))
    return padded

X_sent = torch.tensor([encode_sentence(t, sent_word2idx, MAX_LEN) for t, _ in sentiment_data], dtype=torch.long)
y_sent = torch.tensor([label for _, label in sentiment_data], dtype=torch.long)

# Simple train/test split (small dataset so just 80/20)
split = int(0.8 * len(sentiment_data))
X_train, X_test = X_sent[:split], X_sent[split:]
y_train, y_test = y_sent[:split], y_sent[split:]


class LSTMSentimentClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=1, batch_first=True, bidirectional=True)
        # bidirectional: reads sequence forward AND backward, doubles hidden_dim
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)  # *2 for bidirectional
    
    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, (h_n, _) = self.lstm(embedded)
        # Concatenate last forward and backward hidden states
        # h_n shape: (num_directions * num_layers, batch, hidden_dim)
        h_forward = h_n[0]   # forward direction
        h_backward = h_n[1]  # backward direction
        combined = torch.cat([h_forward, h_backward], dim=1)
        return self.fc(self.dropout(combined))


sent_model = LSTMSentimentClassifier(SENT_VOCAB_SIZE, embed_dim=32, hidden_dim=64).to(device)
optimizer = optim.Adam(sent_model.parameters(), lr=0.005)
criterion = nn.CrossEntropyLoss()

X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

for epoch in range(100):
    sent_model.train()
    optimizer.zero_grad()
    out = sent_model(X_train)
    loss = criterion(out, y_train)
    loss.backward()
    optimizer.step()

sent_model.eval()
with torch.no_grad():
    test_out = sent_model(X_test)
    _, predicted = test_out.max(1)
    accuracy = (predicted == y_test).float().mean().item() * 100

print(f"Sentiment Classifier (Bidirectional LSTM)")
print(f"Test Accuracy: {accuracy:.1f}%")

# Test on new sentences
test_sentences = [
    "the model achieves excellent results on all benchmarks",
    "training failed and loss did not converge at all",
    "decent performance but could be improved further",
]

print("\nPredictions on new sentences:")
sent_model.eval()
with torch.no_grad():
    for sent in test_sentences:
        encoded = torch.tensor([encode_sentence(sent, sent_word2idx, MAX_LEN)], dtype=torch.long).to(device)
        probs = torch.softmax(sent_model(encoded), dim=1)[0]
        pred = probs.argmax().item()
        label = 'POSITIVE' if pred == 1 else 'NEGATIVE'
        confidence = probs[pred].item()
        print(f"  '{sent}'")
        print(f"  -> {label} ({confidence*100:.1f}% confidence)")
        print()

## Real World Problem: Variable Sequence Length at Inference

A fintech company deploys an LSTM-based transaction anomaly detector. It works perfectly in testing. In production, some transactions come in with 200 events in the sequence, while others have just 3. The model was trained on sequences of exactly 50 events.

**What breaks:** They padded short sequences with zeros and truncated long ones during training. In production, they forgot to apply the same padding/truncation. The input shape doesn't match, and the model throws a runtime error 2% of the time — exactly on the high-value transactions with long histories, which are the most important ones to score correctly.

**The actual fix that production teams use:**

1. **Pack padded sequences:** PyTorch's `nn.utils.rnn.pack_padded_sequence` tells the LSTM to ignore padding at the end of sequences. This handles variable lengths without retraining.

2. **Truncation with a sliding window:** For very long sequences, use only the last N events (most recent context is most relevant for anomaly detection).

3. **Store sequence lengths with the model:** Whatever preprocessing you used at training time (truncate to 50, pad to 50) must be part of the serving pipeline as a versioned artifact — not a hardcoded constant someone can accidentally change.

4. **Handle the edge case explicitly:** Add an assertion at the top of your inference function: `assert x.shape[1] == SEQ_LEN, f"Expected {SEQ_LEN}, got {x.shape[1]}"`

This class of bug is responsible for a significant fraction of ML model failures in production. The model is correct. The preprocessing pipeline isn't.

In [ ]:
# ============================================================
# Handling Variable Length Sequences the Right Way
# ============================================================
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

# Simulate batch with variable length sequences
seqs = [
    [1, 2, 3, 4, 5, 6, 7, 8],   # length 8
    [1, 2, 3, 4, 0, 0, 0, 0],   # length 4 (padded)
    [1, 2, 0, 0, 0, 0, 0, 0],   # length 2 (padded)
    [1, 2, 3, 4, 5, 0, 0, 0],   # length 5 (padded)
]
lengths = [8, 4, 2, 5]

# Sort by length descending (required for pack_padded_sequence)
sorted_pairs = sorted(zip(seqs, lengths), key=lambda x: x[1], reverse=True)
seqs_sorted, lengths_sorted = zip(*sorted_pairs)

x = torch.tensor(list(seqs_sorted), dtype=torch.long)
lengths_tensor = torch.tensor(lengths_sorted)

# Build a tiny embedding + LSTM to demo packed sequences
embedding = nn.Embedding(10, 4, padding_idx=0)
lstm = nn.LSTM(4, 8, batch_first=True)

embedded = embedding(x)  # (batch, max_len, embed_dim)

# Without packing: LSTM processes all padding too
out_no_pack, _ = lstm(embedded)

# With packing: LSTM skips padding, each sequence stops at its real length
packed = pack_padded_sequence(embedded, lengths_tensor, batch_first=True)
out_packed, (h_n, _) = lstm(packed)
unpacked, _ = pad_packed_sequence(out_packed, batch_first=True)

print("Variable Length Sequence Handling with pack_padded_sequence")
print("=" * 55)
print(f"Input shape:          {x.shape}  (batch=4, max_len=8)")
print(f"Actual lengths:       {lengths_sorted}")
print(f"\nWithout packing:")
print(f"  LSTM output shape:  {out_no_pack.shape}  (all 8 time steps processed)")
print(f"  Padding positions processed needlessly")
print(f"\nWith packing:")
print(f"  LSTM output shape:  {unpacked.shape}  (same shape, but padding was skipped)")
print(f"  h_n shape:          {h_n.shape}  (final hidden state at each real sequence end)")
print(f"\nThe h_n hidden states represent the actual end of each sequence,")
print(f"not the end of the padding. This is what you want for classification.")

## Interview Corner: MNC-Level Questions

---

**Q1: Explain the vanishing gradient problem in RNNs and how LSTMs solve it.**

*What they're testing:* Core understanding of why LSTMs were invented.

*Answer direction:* In vanilla RNNs, backpropagation through time multiplies the gradient by the weight matrix at every time step. If the spectral radius of the weight matrix is <1, gradients shrink exponentially with sequence length. After 20-30 steps the gradient signal at early time steps is near zero — the network cannot update its weights based on long-range dependencies. LSTMs solve this with the cell state: a separate memory channel that uses additive updates (c_t = f_t * c_prev + i_t * c_hat) instead of the repeated matrix multiplications that cause vanishing. The gradient can flow through the cell state highway without being multiplied to zero.

---

**Q2: What is the purpose of the forget gate bias being initialised to 1?**

*What they're testing:* Depth on LSTM initialisation.

*Answer direction:* The forget gate initialised to 1 (or large positive bias) means the LSTM starts with a strong tendency to remember everything. At the start of training the model hasn't learned what to forget yet, so a default of "keep everything" is safer than a default of "forget everything." If the forget gate started near 0, the cell state would be erased at every step at the beginning of training, and gradients through the cell state highway would be near zero — you'd have the same vanishing gradient problem LSTMs were designed to solve. Jozefowicz et al. (2015) showed empirically that forget gate bias = 1 is consistently one of the most important LSTM tricks.

---

**Q3: When would you use a bidirectional LSTM and what's the tradeoff?**

*What they're testing:* Practical judgment.

*Answer direction:* Bidirectional LSTMs process the sequence in both directions and concatenate the hidden states. This is useful when the full context of a sequence is available at inference time — sentiment classification, named entity recognition, text classification. The word "not" at position 5 affects the meaning of "good" at position 8, but a forward-only LSTM would need to carry that context through 3 more steps. Bidirectional reads "good" with "not" already in the backward hidden state. The tradeoff: bidirectional doubles the hidden dimension (more parameters), and cannot be used for autoregressive generation (you can't read future tokens to predict the current one) or real-time streaming where future context isn't available yet.

---

**Q4: In 2024, when would you still choose an LSTM over a Transformer for a production task?**

*What they're testing:* Current awareness and practical judgment.

*Answer direction:* Four cases: (1) Time series forecasting on tabular sequential data (stock prices, sensor readings) — LSTMs often match Transformers here with much lower compute. (2) On-device or edge inference where model size is constrained — a 2-layer LSTM with 128 hidden units is tiny compared to even a small Transformer. (3) Real-time streaming sequences where you need to process one token at a time without recomputing attention over the entire history — LSTM state is O(1) per step, Transformer attention is O(n²). (4) Applications with very long sequences (>10k steps) where attention is computationally infeasible.

---

**Q5: What is gradient clipping and why is it important for RNN/LSTM training?**

*What they're testing:* Training stability knowledge.

*Answer direction:* Gradient clipping caps the gradient norm to a maximum value (typically 1.0 or 5.0) before the weight update. In RNNs and LSTMs, the same weight matrix appears at every time step. If the gradient norms accumulate, you can get exploding gradients — the gradient magnitude grows exponentially rather than shrinking. Clipping prevents a single large gradient from causing a destructive weight update that blows up training. In PyTorch: `torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)`. Always call this after `loss.backward()` and before `optimizer.step()`. It's standard practice for any recurrent model.

## ML Spotlight

**Mamba (State Space Models, 2023)**

Mamba is the most significant alternative to both LSTMs and Transformers for sequential data. It's based on State Space Models (SSMs) and achieves linear O(n) scaling with sequence length (vs O(n²) for Transformers) while matching Transformer quality on language tasks.

Why it's relevant to LSTMs: Mamba can be seen as a hardware-efficient, theoretically grounded evolution of the LSTM idea — a selective state that decides what to remember and forget, but with better theoretical properties and GPU efficiency.

In production as of 2024: Mamba is being integrated into hybrid architectures (Jamba = Mamba + Transformer) for long-context tasks where pure Transformers are too slow.

Paper: [Mamba: Linear-Time Sequence Modeling with Selective State Spaces](https://arxiv.org/abs/2312.00752)

GitHub: https://github.com/state-spaces/mamba

## Practice Exercise

1. Modify the language model to use GRU instead of LSTM (`nn.GRU` has the same API as `nn.LSTM`). GRU has fewer gates (no separate cell state) — compare training speed and final perplexity.

2. Try different sequence lengths (SEQ_LEN = 3, 6, 10) for the language model. Does longer context improve perplexity?

3. For the sentiment classifier, add an attention mechanism on top of the LSTM:
```python
# After lstm_out: (batch, seq_len, hidden*2)
attn_weights = torch.softmax(lstm_out @ attn_vector, dim=1)  # (batch, seq_len)
context = (attn_weights.unsqueeze(-1) * lstm_out).sum(1)     # weighted sum
```
Does attention improve accuracy on the test set?

---

**What's Next**

Day 30: Transformers — Self-attention from scratch, why it replaced RNNs for most NLP tasks, and running BERT for text classification in 15 lines using HuggingFace.